[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/03_minimal_ai_systems_overview.ipynb)

# 03. Minimal AI systems overview — architecture → training → inference

목표는 **크기만 줄이고 원형의 중요한 계산 그래프를 보존한 뒤, 학습에서 끝내지 않고 실제 추론/샘플링까지 연결**하는 것이다.

각 모델은 `architecture/forward → tiny training → inference/sampling → 구조 검증` 순서로 본다.


In [ ]:
import importlib.util
import sys
import urllib.request

import torch
import torch.nn.functional as F

MODEL_URL = (
    "https://raw.githubusercontent.com/"
    "HisameOgasahara/deep-learning-diagnostics-and-improvement/"
    "main/practice/paper_faithful_tiny_models.py"
)
LOCAL_MODEL_PATH = "/tmp/paper_faithful_tiny_models.py"

urllib.request.urlretrieve(MODEL_URL, LOCAL_MODEL_PATH)

spec = importlib.util.spec_from_file_location(
    "tiny_models",
    LOCAL_MODEL_PATH,
)
tiny_models = importlib.util.module_from_spec(spec)
sys.modules["tiny_models"] = tiny_models
spec.loader.exec_module(tiny_models)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
torch.manual_seed(7)

print("device:", device)
print("torch:", torch.__version__)


## A. Tiny GPT-2-like decoder

구조는 learned token/position embedding → pre-norm causal self-attention → MLP residual → tied LM head다. 학습은 next-token prediction, 추론은 autoregressive generation으로 연결한다.


In [ ]:
gpt = tiny_models.TinyGPT().to(device)
tokens = torch.tensor(
    [[1, 2, 3, 4, 5, 6, 7, 8]],
    device=device,
)
optimizer = torch.optim.AdamW(gpt.parameters(), lr=3e-3)

for step in range(4):
    input_tokens = tokens[:, :-1]
    target_tokens = tokens[:, 1:]
    logits = gpt(input_tokens)
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_tokens.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("GPT train", step, "loss", round(loss.item(), 4))


In [ ]:
@torch.no_grad()
def greedy_generate(model, prompt_tokens, new_tokens=4):
    generated = prompt_tokens.clone()

    for _ in range(new_tokens):
        logits = model(generated)
        next_token = logits[:, -1].argmax(
            dim=-1,
            keepdim=True,
        )
        generated = torch.cat(
            [generated, next_token],
            dim=1,
        )

    return generated

prompt = torch.tensor([[1, 2, 3]], device=device)
generated = greedy_generate(gpt, prompt, new_tokens=4)
print("prompt:", prompt)
print("generated:", generated)

with torch.no_grad():
    _, attention_maps = gpt(
        tokens[:, :-1],
        return_attn=True,
    )

sequence_length = tokens.size(1) - 1
future_positions = torch.triu(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool,
        device=device,
    ),
    diagonal=1,
)
future_mass = attention_maps[0][..., future_positions].sum()
print("future attention mass:", float(future_mass))


## B. Tiny ViT

image → patches → patch projection → CLS + learned position → Transformer encoder → classifier를 유지한다. 학습 뒤 classifier inference 결과를 직접 출력한다.


In [ ]:
vit = tiny_models.TinyViT().to(device)
images = torch.randn(4, 3, 16, 16, device=device)
labels = torch.tensor([0, 1, 2, 1], device=device)
optimizer = torch.optim.AdamW(vit.parameters(), lr=3e-3)

for step in range(4):
    logits = vit(images)
    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("ViT train", step, "loss", round(loss.item(), 4))

with torch.no_grad():
    logits, attention_maps = vit(
        images,
        return_attn=True,
    )
    predictions = logits.argmax(dim=-1)

print("ViT predictions:", predictions)
print("attention shape:", tuple(attention_maps[0].shape))


## C. Tiny 2D DiT + Flow Matching

DiT의 patch token, fixed 2D position, timestep embedding, adaLN-Zero, conditioned final layer를 유지한다. 학습 뒤 noise에서 velocity field를 적분해 sample trajectory를 만든다.


In [ ]:
dit2 = tiny_models.Tiny2DDiT().to(device)
x_data = torch.randn(3, 2, 8, 8, device=device)
x_noise = torch.randn_like(x_data)
optimizer = torch.optim.AdamW(dit2.parameters(), lr=3e-3)

for step in range(6):
    t = torch.rand(3, device=device)
    t_broadcast = t[:, None, None, None]
    x_t = (1 - t_broadcast) * x_data + t_broadcast * x_noise
    target_velocity = x_noise - x_data
    predicted_velocity = dit2(x_t, t)
    loss = F.mse_loss(predicted_velocity, target_velocity)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("2D DiT-flow train", step, "loss", round(loss.item(), 4))


In [ ]:
@torch.no_grad()
def sample_2d_flow(model, initial_noise, num_steps=6):
    sample = initial_noise.clone()
    trajectory = [sample.clone()]
    dt = 1.0 / num_steps

    for step in range(num_steps):
        t_value = 1.0 - step * dt
        t = torch.full(
            (sample.size(0),),
            t_value,
            device=sample.device,
        )
        velocity = model(sample, t)
        sample = sample - dt * velocity
        trajectory.append(sample.clone())

    return sample, trajectory

sample_2d, trajectory_2d = sample_2d_flow(dit2, x_noise[:1])
print("2D sample shape:", tuple(sample_2d.shape))
for index, state in enumerate(trajectory_2d):
    print("2D sample", index, "mean", round(state.mean().item(), 4))


## D. Tiny 3D DiT + Flow

같은 DiT 원리를 3D non-overlapping patch grid에 적용하고, 학습 뒤 3D noise volume을 velocity-field 적분으로 변환한다.


In [ ]:
dit3 = tiny_models.Tiny3DDiT().to(device)
volume_data = torch.randn(3, 1, 4, 4, 4, device=device)
volume_noise = torch.randn_like(volume_data)
optimizer = torch.optim.AdamW(dit3.parameters(), lr=3e-3)

for step in range(6):
    t = torch.rand(3, device=device)
    t_broadcast = t[:, None, None, None, None]
    volume_t = (
        (1 - t_broadcast) * volume_data
        + t_broadcast * volume_noise
    )
    target_velocity = volume_noise - volume_data
    predicted_velocity = dit3(volume_t, t)
    loss = F.mse_loss(predicted_velocity, target_velocity)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("3D DiT-flow train", step, "loss", round(loss.item(), 4))

with torch.no_grad():
    sample = volume_noise[:1].clone()
    num_steps = 6
    dt = 1.0 / num_steps

    for step in range(num_steps):
        t_value = 1.0 - step * dt
        t = torch.full((1,), t_value, device=device)
        sample = sample - dt * dit3(sample, t)
        print("3D sample", step, "mean", round(sample.mean().item(), 4))


## E. Tiny π0-like VLA Flow Policy

vision+language prefix와 state+noisy-action suffix를 같은 masked Transformer에 넣는다. 학습 뒤 noise action chunk에서 velocity를 적분해 최종 continuous action chunk를 만든다.


In [ ]:
policy = tiny_models.TinyVLAFlowPolicy().to(device)
batch_size = 3
vision = torch.randn(batch_size, 3, 16, 16, device=device)
language = torch.tensor(
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]],
    device=device,
)
robot_state = torch.randn(batch_size, 4, device=device)
action_data = torch.randn(batch_size, 4, 4, device=device)
action_noise = torch.randn_like(action_data)
optimizer = torch.optim.AdamW(policy.parameters(), lr=3e-3)

for step in range(6):
    t = torch.rand(batch_size, device=device)
    t_broadcast = t[:, None, None]
    action_t = (
        (1 - t_broadcast) * action_data
        + t_broadcast * action_noise
    )
    target_velocity = action_noise - action_data
    predicted_velocity = policy(
        vision,
        language,
        robot_state,
        action_t,
        t,
    )
    loss = F.mse_loss(predicted_velocity, target_velocity)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("VLA-flow train", step, "loss", round(loss.item(), 4))


In [ ]:
with torch.no_grad():
    action = action_noise[:1].clone()
    num_steps = 6
    dt = 1.0 / num_steps

    for step in range(num_steps):
        t_value = 1.0 - step * dt
        t = torch.full((1,), t_value, device=device)
        velocity = policy(
            vision[:1],
            language[:1],
            robot_state[:1],
            action,
            t,
        )
        action = action - dt * velocity

    _, attention_maps, attention_mask = policy(
        vision[:1],
        language[:1],
        robot_state[:1],
        action_noise[:1],
        torch.ones(1, device=device),
        return_attn=True,
    )

print("sampled action chunk shape:", tuple(action.shape))
print(action)
print("joint attention shape:", tuple(attention_maps[0].shape))
print("prefix -> action allowed:", bool(attention_mask[0, -1]))
print("action -> prefix allowed:", bool(attention_mask[-1, 0]))


## References and provenance

**GPT** — Radford et al., GPT/GPT-2; Vaswani et al., Transformer.

**ViT** — Dosovitskiy et al., *An Image is Worth 16x16 Words*.

**DiT** — Peebles & Xie, *Scalable Diffusion Models with Transformers* 및 공식 DiT 구현.

**Flow Matching** — Lipman et al., *Flow Matching for Generative Modeling*.

**VLA / π0** — Physical Intelligence π0 및 공식 `openpi` 구현.

이 노트북은 각 구조를 학습 loss에서 끝내지 않고 **autoregressive generation / classifier inference / ODE-style flow sampling / action-flow sampling**까지 연결한다.
